# 28. Air-to-ground coupling of the explosion airwave

At each site with an infraBSU and a Trillium side by side, this notebook measures how the airwave drives
the ground. The source is a single impulsive airwave, which a launch never provides:

* **scalar coupling** (vertical velocity per pressure, µm s⁻¹ Pa⁻¹, 1–40 Hz): RMS and peak ratios, as in
  Artemis II notebook 50 and AMOS-6 (2.31 µm s⁻¹ Pa⁻¹ at BCHH, 1.4 km);
* **compliance**: the regression of vertical ground *displacement* on pressure, $u_z = C\,p$. For
  quasi-static loading by the airwave, compression pushes the ground down, so $C < 0$ and $v_z = C\,dp/dt$;
* **transfer functions** $H_v(f) = S_{pv}/S_{pp}$, coherence, and the spectral compliance
  $H_u = H_v / (i 2\pi f)$;
* the **ground resonance** excited by the shock and what it, with the low-frequency compliance, implies
  for the near-surface layer;
* **horizontal to vertical** ratios of the air-coupled motion;
* a **same-site comparison with Artemis II** (B05 and B29, the same sensor pairs, with the same code).

Pairs: B20 DD1/DHZ (100 samples/s; 0.5–12 Hz because of the machinery tones at 15 and 35 Hz), B14
DD0/DD1/DD2 with DHZ (three infraBSUs on one Trillium), B29 DD0/DHZ and B05 DD0/DHZ. Everything is
taken to 100 samples/s. Filters are causal, so the shock does not ring ahead of itself.

Output (`cfg.OUT_DIR / "28_air_ground_coupling"`): `coupling_metrics.csv`, `coupling_transfer_functions.csv`,
`coupling_waveforms.png`, `coupling_transfer_functions.png`, `coupling_summary.png`,
`layer_resonance.csv`, `horizontal_vertical.csv`, `artemis2_comparison.csv`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import csd, welch

import ksclib
import newglenn_config as cfg

warnings.filterwarnings('ignore', message='ObsPy can not map unit')
ORIGIN = cfg.ORIGIN
OUT = cfg.output_dir('28_air_ground_coupling')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})
stations = cfg.station_table()
air = cfg.airwave_speeds()
FS = 100.0
PRE, POST = 1.0, 19.0                 # airwave window: 1 s before to 19 s after the predicted arrival
NOISE = (-120.0, -20.0)               # s after origin: before the seismic arrivals
SEG_S = 2.0                           # Welch segment
COH_MIN = 0.5
AMOS6_K = 2.31                        # um/s/Pa, AMOS-6 broadband median at BCHH (1.4 km)
PAIRS = [('1R.B20.00.DD1', '1R.B20.00.DHZ', (0.5, 12.0)),
         ('1R.B14.00.DD0', '1R.B14.00.DHZ', (0.5, 40.0)),
         ('1R.B14.00.DD1', '1R.B14.00.DHZ', (0.5, 40.0)),
         ('1R.B14.00.DD2', '1R.B14.00.DHZ', (0.5, 40.0)),
         ('1R.B29.00.DD0', '1R.B29.00.DHZ', (0.5, 40.0)),
         ('1R.B05.00.DD0', '1R.B05.00.DHZ', (0.5, 40.0))]
raw = cfg.event_stream_raw(starttime=ORIGIN - 300, endtime=ORIGIN + 300)
st = cfg.instrument_correct(raw)      # polarity_reversed infraBSUs flipped
print(sorted(tr.id for tr in st))

## Helpers

In [ ]:
def at_fs(tr, fs=FS):
    t = tr.copy()
    if t.stats.sampling_rate > fs:
        t.filter('lowpass', freq=0.4 * fs, corners=8, zerophase=True)
        t.resample(fs, no_filter=True)
    return t

def band(tr, lo, hi):
    return tr.copy().filter('bandpass', freqmin=lo, freqmax=min(hi, 0.45 * tr.stats.sampling_rate),
                            corners=4, zerophase=False)

def common(a, b, t0, t1):
    x, y = a.slice(t0, t1).data.astype(float), b.slice(t0, t1).data.astype(float)
    n = min(len(x), len(y))
    return x[:n], y[:n]

def lag_regression(x, y, fs, max_lag=0.3):
    """y ~ k x(t - lag): k, lag (s, y later if positive), correlation at the lag of largest |r|."""
    n, nl, best = len(x), int(max_lag * fs), (0.0, 0.0, 0.0)
    for k in range(-nl, nl + 1):
        a, b = x[max(0, -k):n - max(0, k)], y[max(0, k):n - max(0, -k)]
        r = np.corrcoef(a, b)[0, 1]
        if abs(r) > abs(best[2]):
            best = (np.dot(a, b) / np.dot(a, a), k / fs, r)
    return best

def spectra(x, y, fs, seg_s=SEG_S):
    n = int(seg_s * fs)
    f, pxx = welch(x, fs, nperseg=n)
    _, pyy = welch(y, fs, nperseg=n)
    _, pxy = csd(x, y, fs, nperseg=n)
    return f, pxx, pyy, pxy

def pair_metrics(p_raw, v_raw, t_air, origin, lo, hi, broadband=(1.0, 40.0)):
    """Scalar coupling, compliance and spectra for one pressure/velocity pair (traces in Pa and m/s)."""
    p, v = at_fs(p_raw), at_fs(v_raw)
    bb = (broadband[0], min(broadband[1], hi))
    pb, vb = band(p, *bb), band(v, *bb)
    w = (origin + t_air - PRE, origin + t_air + POST)
    nw = (origin + NOISE[0], origin + NOISE[1])
    x, y = common(pb, vb, *w)
    xn, yn = common(pb, vb, *nw)
    k_v, lag_v, r_v = lag_regression(x, y, FS)          # velocity on pressure, as for Artemis II and AMOS-6
    pc, vc = band(p, lo, hi), band(v, lo, hi)
    u = vc.copy().integrate()
    xc, uc = common(pc, u, *w)
    k_c, lag_c, r_c = lag_regression(xc, uc, FS)
    xr, yr = common(p, v, *w)
    f, pxx, pyy, pxy = spectra(xr - xr.mean(), yr - yr.mean(), FS)
    H = pxy / pxx
    coh = np.abs(pxy) ** 2 / (pxx * pyy)
    return dict(k_rms_um_s_Pa=1e6 * y.std() / x.std(), k_peak_um_s_Pa=1e6 * np.abs(y).max() / np.abs(x).max(),
                peak_p_Pa=np.abs(x).max(), peak_v_um_s=1e6 * np.abs(y).max(),
                snr_v=y.std() / yn.std(), noise_k_rms_um_s_Pa=1e6 * yn.std() / xn.std(),
                compliance_nm_Pa=1e9 * k_c, compliance_lag_s=lag_c, compliance_r=r_c,
                v_on_p_um_s_Pa=1e6 * k_v, v_on_p_r=r_v,
                band=f'{bb[0]:g}-{bb[1]:g}', compliance_band=f'{lo:g}-{hi:g}'), (f, H, coh)

## Coupling at each pair

`k_rms`/`k_peak` are in the broadband (1–40 Hz; B20 1–12 Hz), comparable with Artemis II and AMOS-6.
`compliance` is the regression of displacement on pressure (0.5–40 Hz; B20 0.5–12 Hz) in the airwave
window, with its lag and correlation; `v_on_p_r` is the correlation of velocity with pressure (broadband, best
lag), the regression used for Artemis II and AMOS-6.

In [ ]:
rows, tfs = [], {}
for p_id, v_id, (lo, hi) in PAIRS:
    sta = p_id.split('.')[1]
    m, (f, H, coh) = pair_metrics(st.select(id=p_id)[0], st.select(id=v_id)[0], air.travel_time_s[sta], ORIGIN, lo, hi)
    pair = f"{p_id.replace('1R.', '')} / {v_id.split('.')[-1]}"
    rows.append(dict(pair=pair, station=sta, distance_km=stations.distance_km[sta], **m))
    tfs[pair] = dict(f=f, H=H, coh=coh, hi=hi, dist=stations.distance_km[sta])
coup = pd.DataFrame(rows).sort_values(['distance_km', 'pair']).reset_index(drop=True)
for lo, hi in ((1, 3), (3, 6), (6, 12), (12, 40)):
    for i, r in coup.iterrows():
        d = tfs[r.pair]
        mk = (d['f'] >= lo) & (d['f'] < min(hi, d['hi'])) & (d['coh'] >= COH_MIN)
        coup.loc[i, f'Hv_{lo}-{hi}Hz_um_s_Pa'] = 1e6 * np.median(np.abs(d['H'][mk])) if mk.any() else np.nan
        coup.loc[i, f'coh_{lo}-{hi}Hz'] = np.median(d['coh'][(d['f'] >= lo) & (d['f'] < min(hi, d['hi']))]) if lo < d['hi'] else np.nan
coup.to_csv(OUT / 'coupling_metrics.csv', index=False)
tf_rows = [dict(pair=k, f_hz=fi, Hv_um_s_Pa=1e6 * abs(h), phase_deg=np.degrees(np.angle(h)), coherence=c)
           for k, d in tfs.items() for fi, h, c in zip(d['f'], d['H'], d['coh'])]
pd.DataFrame(tf_rows).to_csv(OUT / 'coupling_transfer_functions.csv', index=False)
show = ['pair', 'distance_km', 'peak_p_Pa', 'peak_v_um_s', 'k_rms_um_s_Pa', 'k_peak_um_s_Pa', 'snr_v',
        'v_on_p_r', 'compliance_nm_Pa', 'compliance_lag_s', 'compliance_r']
display(coup[show].round(3))
display(coup[['pair'] + [c for c in coup.columns if c.startswith(('Hv_', 'coh_'))]].round(2))

### Waveforms: displacement follows pressure

For each station: pressure (black), and vertical displacement divided by the fitted compliance (orange),
so the two overlap where the ground responds quasi-statically to the load. Velocity (grey, scaled) is
shown for reference. Band as for the compliance; times after the predicted airwave arrival.

In [ ]:
show_pairs = [p for p in PAIRS if p[0] != '1R.B14.00.DD1' and p[0] != '1R.B14.00.DD2']
fig, axs = plt.subplots(len(show_pairs), 1, figsize=(11, 2.3 * len(show_pairs)), sharex=True)
for ax, (p_id, v_id, (lo, hi)) in zip(axs, show_pairs):
    sta = p_id.split('.')[1]
    ta = air.travel_time_s[sta]
    r = coup[coup.pair.str.startswith(p_id.replace('1R.', ''))].iloc[0]
    p = band(at_fs(st.select(id=p_id)[0]), lo, hi)
    v = band(at_fs(st.select(id=v_id)[0]), lo, hi)
    u = v.copy().integrate()
    w = (ORIGIN + ta - 2, ORIGIN + ta + 8)
    t = lambda tr: tr.slice(*w).times() - 2  # noqa: E731
    ps = p.slice(*w).data
    ax.plot(t(v), v.slice(*w).data / np.abs(v.slice(*w).data).max() * np.abs(ps).max(), color='#bbbbbb', lw=0.6, label='v$_z$ (scaled)')
    ax.plot(t(p), ps, color='#222222', lw=0.8, label='pressure')
    ax.plot(t(u), u.slice(*w).data / (r.compliance_nm_Pa * 1e-9), color='#eb6834', lw=0.9, label='u$_z$ / C')
    ax.set_title(f"{sta} {stations.distance_km[sta]:.1f} km: C = {r.compliance_nm_Pa:.0f} nm/Pa, r = {r.compliance_r:+.2f} "
                 f"({lo:g}-{hi:g} Hz)", loc='left', fontsize=9)
    ax.set_ylabel('Pa')
axs[0].legend(fontsize=7, ncol=3, loc='upper right')
axs[-1].set_xlabel('Time after predicted airwave arrival (s)')
fig.tight_layout()
fig.savefig(OUT / 'coupling_waveforms.png', dpi=150, bbox_inches='tight')

## Transfer functions, spectral compliance and a layer resonance

The spectral compliance $|C(f)| = |H_v|/(2\pi f)$ is compared with two simple models:

* **elastic half-space** under a load sweeping across it at the airwave speed: $|C| \propto 1/f$
  (dashed line);
* **soft layer over stiffer ground**: $|C|$ flat at long wavelengths, $C_0 = H/M$ for a layer of thickness
  $H$ and constrained modulus $M = \rho V_P^2$, with a vertical resonance near $f_0 = V_P/(4H)$. The
  resonance shows as the ringing of the ground after the shock (waveform figure above).

$f_0$ is measured as the spectral peak (2–15 Hz) of the vertical velocity from 2 to 10 s after the airwave
arrival. $C_0$ is the median coherent $|C|$ at 0.5–2.5 Hz. For a uniform layer over a rigid base, the two
give $H = 1/(16\,\rho\,C_0\,f_0^2)$ and $V_P = 4 H f_0$. With $\rho = 1800$ kg m⁻³ this is a
one-dimensional, order-of-magnitude reading of the near-surface structure, to be tested against borehole
or refraction data.

In [ ]:
RHO = 1800.0
fig, axs = plt.subplots(3, 1, figsize=(8, 10), sharex=True)
lay = []
for pair, d in tfs.items():
    f, H, coh = d['f'], d['H'], d['coh']
    sta = pair.split('.')[0]
    m = (f >= 0.5) & (f <= d['hi'])
    good = m & (coh >= COH_MIN)
    Cf = np.abs(H) / (2 * np.pi * np.where(f > 0, f, np.nan))
    l, = axs[0].loglog(f[good], 1e6 * np.abs(H[good]), '.', ms=3, label=f"{pair} ({d['dist']:.1f} km)")
    axs[1].loglog(f[good], 1e9 * Cf[good], '.', ms=3, color=l.get_color())
    axs[2].semilogx(f[m], coh[m], lw=0.8, color=l.get_color())
    lowb = good & (f <= 2.5)
    C0 = np.median(Cf[lowb]) if lowb.any() else np.nan
    z = band(at_fs(st.select(station=sta, channel='DHZ')[0]), 1.0, min(20.0, d['hi']))
    ta = air.travel_time_s[sta]
    x = z.slice(ORIGIN + ta + 2, ORIGIN + ta + 10).data
    fz, pz = welch(x - x.mean(), FS, nperseg=int(4 * FS))
    mz = (fz >= 2) & (fz <= min(15, d['hi']))
    f0 = fz[mz][np.argmax(pz[mz])]
    Hm = 1 / (16 * RHO * C0 * f0 ** 2) if np.isfinite(C0) else np.nan
    lay.append(dict(pair=pair, C0_nm_Pa=1e9 * C0, n_low=int(lowb.sum()), f0_hz=f0, layer_H_m=Hm, layer_Vp_m_s=4 * Hm * f0))
    axs[1].axvline(f0, color=l.get_color(), lw=0.6, ls=':')
ff = np.logspace(0, 1.6, 20)
axs[1].loglog(ff, 100 / ff, '--', color='#999999', lw=1, label=r'half-space, $\propto 1/f$')
axs[0].axhline(AMOS6_K, color='#cc0000', ls='--', lw=0.8, label=f'AMOS-6 ({AMOS6_K} µm/s/Pa)')
axs[0].set_ylabel('|H$_v$| (µm s$^{-1}$ Pa$^{-1}$)')
axs[1].set_ylabel('|C(f)| (nm Pa$^{-1}$)')
axs[2].set(ylabel='Coherence', xlabel='Frequency (Hz)', ylim=(0, 1.02))
axs[2].axhline(COH_MIN, color='#999999', ls=':', lw=0.8)
axs[0].legend(fontsize=7)
axs[1].legend(fontsize=7)
axs[0].set_title(f'Airwave window ({PRE:g} s before to {POST:g} s after arrival); points where coherence >= {COH_MIN}; '
                 'dotted: ringing frequency f$_0$', loc='left', fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'coupling_transfer_functions.png', dpi=150, bbox_inches='tight')
lay = pd.DataFrame(lay).set_index('pair')
lay.to_csv(OUT / 'layer_resonance.csv')
display(lay.round(1))

## Horizontal to vertical ratio of the air-coupled motion

RMS of each horizontal over RMS of the vertical in the airwave window (1–40 Hz; B20 1–12 Hz), for
stations with working horizontals. DH1/DH2 are as installed (not rotated).

In [ ]:
hv = []
for sta, hi in (('B20', 12.0), ('B14', 40.0), ('B29', 40.0)):
    ta = air.travel_time_s[sta]
    w = (ORIGIN + ta - PRE, ORIGIN + ta + POST)
    z = band(at_fs(st.select(station=sta, channel='DHZ')[0]), 1.0, hi).slice(*w).data
    r = dict(station=sta, distance_km=stations.distance_km[sta])
    for cha in ('DH1', 'DH2'):
        h = band(at_fs(st.select(station=sta, channel=cha)[0]), 1.0, hi).slice(*w).data
        r[f'{cha}/DHZ'] = h.std() / z.std()
    hv.append(r)
hv = pd.DataFrame(hv).set_index('station')
hv.to_csv(OUT / 'horizontal_vertical.csv')
display(hv.round(2))

## Same sites, different source: Artemis II

The same pairs at B05 and B29 during the Artemis II launch (0–180 s after launch, as in Artemis II
notebook 50), with the same functions. Artemis II's B29 DHZ comes from the restored SDS folder; unreadable
SDS day files are skipped and listed (`ksclib.read_sds_files`). For a
launch the airwave window is the whole 180 s, and the noise window is 900–720 s before launch.

In [ ]:
a2 = ksclib.load_event(cfg.PAPER_DIR.parent / '09_artemis2_paper' / 'artemis2.yaml')
L = a2.origin_time
a2st = a2c = None
from obspy import Stream
a2st = Stream()
for sta in ('B05', 'B29'):
    for cha in ('DD0', 'DHZ'):
        a2st += ksclib.read_sds_files(a2.master_sds, '1R', sta, '00', cha, L - 1000, L + 400)
for sid in a2.restored:
    if not a2st.select(id=sid):
        a2st += ksclib.read_restored(a2, sid, L - 1000, L + 400)
a2st.merge()
print(a2st)
print('skipped:', ksclib.SKIPPED_FILES)
a2c = ksclib.instrument_correct(a2st, a2)

def a2_metrics(p_tr, v_tr):
    p, v = at_fs(p_tr), at_fs(v_tr)
    pb, vb = band(p, 1, 40), band(v, 1, 40)
    x, y = common(pb, vb, L, L + 180)
    xn, yn = common(pb, vb, L - 900, L - 720)
    pc, vc = band(p, 0.5, 40), band(v, 0.5, 40)
    xc, uc = common(pc, vc.copy().integrate(), L, L + 180)
    k_c, lag_c, r_c = lag_regression(xc, uc, FS)
    return dict(k_rms_um_s_Pa=1e6 * y.std() / x.std(), k_peak_um_s_Pa=1e6 * np.abs(y).max() / np.abs(x).max(),
                peak_p_Pa=np.abs(x).max(), snr_v=y.std() / yn.std(), compliance_nm_Pa=1e9 * k_c,
                compliance_lag_s=lag_c, compliance_r=r_c)

cmp_rows = []
for sta in ('B05', 'B29'):
    m = a2_metrics(a2c.select(station=sta, channel='DD0')[0], a2c.select(station=sta, channel='DHZ')[0])
    cmp_rows.append(dict(source='Artemis II launch', station=sta, **m))
    r = coup[coup.station == sta].iloc[0]
    cmp_rows.append(dict(source='New Glenn explosion', station=sta,
                         **{k: r[k] for k in m}))
cmp = pd.DataFrame(cmp_rows)
cmp.to_csv(OUT / 'artemis2_comparison.csv', index=False)
display(cmp.round(3))

## Summary figure

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for _, r in coup.iterrows():
    axs[0].plot(r.distance_km, r.k_rms_um_s_Pa, 'o', color='#2a78d6')
    axs[0].plot(r.distance_km, r.k_peak_um_s_Pa, 's', mfc='none', color='#2a78d6')
    axs[1].plot(r.distance_km, abs(r.compliance_nm_Pa), 'o', color='#eb6834' if r.compliance_nm_Pa < 0 else '#999999')
for sta, g in coup.groupby('station'):
    axs[0].annotate(sta, (g.distance_km.iloc[0], g.k_rms_um_s_Pa.max()), xytext=(4, 3), textcoords='offset points', fontsize=8)
    axs[1].annotate(sta, (g.distance_km.iloc[0], abs(g.compliance_nm_Pa).max()), xytext=(4, 3), textcoords='offset points', fontsize=8)
for _, r in cmp[cmp.source == 'Artemis II launch'].iterrows():
    d = stations.distance_km[r.station]
    axs[0].plot(d, r.k_rms_um_s_Pa, '^', color='#1baf7a')
    axs[1].plot(d, abs(r.compliance_nm_Pa), '^', color='#1baf7a')
axs[0].axhline(AMOS6_K, color='#cc0000', ls='--', lw=0.8)
axs[0].set(yscale='log', xlabel='Distance from LC-36 (km)', ylabel='Coupling (µm s$^{-1}$ Pa$^{-1}$)',
           title='RMS (filled) and peak (open) coupling; Artemis II RMS (triangles); AMOS-6 (dashed)')
axs[1].set(yscale='log', xlabel='Distance from LC-36 (km)', ylabel='|C| (nm Pa$^{-1}$)',
           title='Compliance (orange: C < 0, loading; grey: C > 0); Artemis II (triangles)')
for ax in axs:
    ax.title.set_fontsize(9)
fig.tight_layout()
fig.savefig(OUT / 'coupling_summary.png', dpi=150, bbox_inches='tight')

## What this shows

* **The airwave loads the ground quasi-statically.** Vertical *displacement* follows pressure with no
  measurable lag (≤ 0.02 s), with the sign of loading: compression pushes the ground down. Correlations are
  −0.75 to −0.76 at B14 (all three infraBSUs), −0.54 at B29 and −0.40 at B05. Velocity therefore follows
  −C dp/dt, which is why regressing velocity directly on pressure (as for Artemis II and AMOS-6) correlates
  poorly here (|r| ≈ 0.3).
* **Compliance.** C ≈ −75 nm/Pa at B14 (−60 to −78 across the three infraBSUs, a spread that reflects their
  relative calibration), −68 nm/Pa at B29 and −17 nm/Pa at B05.
* **Same sites, two sources.** For Artemis II, the same code gives −15.9 nm/Pa at B05 and −65.9 nm/Pa at
  B29, against −16.8 and −68.1 for the explosion: within 6%. The RMS coupling differs by a factor of two
  between the sources (B05 3.2 vs 1.8; B29 10.5 vs 5.3 µm s⁻¹ Pa⁻¹), because it depends on the source
  spectrum. Compliance is a property of the site, and the better quantity to compare between events and
  with models.
* **Scalar coupling.** RMS 1.8–5.3 µm s⁻¹ Pa⁻¹ and peak 1.1–3.9 µm s⁻¹ Pa⁻¹ at 1–40 Hz, the same order as
  AMOS-6 (2.31 µm s⁻¹ Pa⁻¹ at BCHH).
* **Spectral compliance and resonance.** |C(f)| is flat at 70–110 nm/Pa below 2.5 Hz (22 nm/Pa at B05),
  peaks near the ground resonance, and falls above about 15 Hz. That is not the 1/f trend of a uniform
  half-space. The shock sets the ground ringing at 4.5 Hz (B14), 5.5 Hz (B29) and 4.8 Hz (B05), the band
  where AMOS-6 coupling peaked at BCHH (4–5 Hz). Read as a uniform soft layer over a stiff base, B14 and
  B29 imply a layer 15–25 m thick with Vp ≈ 330–440 m/s. B05 is much stiffer, and the simple layer model
  suits it less. These are order-of-magnitude readings to test against borehole or refraction data.
* **Horizontal/vertical.** The air-coupled motion has H/V ≈ 1 at B29 and ≈ 0.45 at B14 and B20.
* **B20.** The compliance there has the opposite sign (+28 nm/Pa, r = +0.48), coherence is low, and the
  Trillium is noisy at 100 samples/s. Either B20's vertical polarity is reversed or the record is too noisy
  to tell. Check the installation before B20 is used for polarity-sensitive work.
* **Check.** The Artemis II B29 compliance lag (0.24 s) is near the edge of the ±0.3 s search; the restored
  B29 DHZ timing is worth checking.